In [2]:
import json
import logging
from pathlib import Path
from typing import List, Dict, Set
from dataclasses import dataclass
from tqdm import tqdm

In [4]:
"""
Biblical Content Filtering Pipeline - JSONL Output
Filters BDC and PSC chunks against Vulgate corpus using Overlap Coefficient
Outputs JSONL 
"""

import json
import logging
from pathlib import Path
from typing import List, Dict, Set
from dataclasses import dataclass
from tqdm import tqdm

# Configuration
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

OVERLAP_THRESHOLD = 0.75
DATA_DIR = Path("../data")


@dataclass
class FilterStats:
    """Statistics for biblical filtering results."""
    total_chunks: int = 0
    biblical_chunks: int = 0
    filtered_chunks: int = 0
    
    @property
    def biblical_percentage(self) -> float:
        return (self.biblical_chunks / self.total_chunks * 100) if self.total_chunks > 0 else 0.0
    
    @property
    def retention_rate(self) -> float:
        return (self.filtered_chunks / self.total_chunks) if self.total_chunks > 0 else 0.0


class BiblicalFilter:
    """
    Uses Szymkiewicz–Simpson coefficient to detect pure biblical quotations
    """
    
    def __init__(self, vulgate_chunks: List[Dict], threshold: float = OVERLAP_THRESHOLD):
        """
        Initialize filter with pre-computed Vulgate lemma sets.
        
        Args:
            vulgate_chunks: List of Vulgate verse chunks with lemmatized field
            threshold: Overlap threshold for flagging biblical content (default: 0.75)
        """
        self.threshold = threshold
        
        # Pre-compute Vulgate lemma sets for fast lookup
        logger.info(f"Pre-computing Vulgate lemma sets from {len(vulgate_chunks)} verses...")
        self.vulgate_sets = {}
        
        for verse in tqdm(vulgate_chunks, desc="Building Vulgate index"):
            verse_id = verse["chunk_id"]
            lemmas = verse.get("lemmatized", [])
            
            # Remove punctuation and convert to lowercase set
            clean_lemmas = [l.lower() for l in lemmas if l and l.isalpha()]
            self.vulgate_sets[verse_id] = set(clean_lemmas)
        
        logger.info(f"✓ Indexed {len(self.vulgate_sets)} Vulgate verses\n")
    
    def overlap_coefficient(self, set1: Set[str], set2: Set[str]) -> float:
        """
        Compute Overlap Coefficient (Szymkiewicz–Simpson coefficient).
        
        Measures the overlap between two sets normalized by the smaller set.
        Value = 1.0 indicates the smaller set is fully contained in the larger.
        Handles size asymmetry (e.g., short Bible verse in long commentary chunk).

        """
        if not set1 or not set2:
            return 0.0
        
        intersection = len(set1 & set2)
        min_size = min(len(set1), len(set2))
        
        return intersection / min_size
    
    def filter_chunk(self, chunk_tokens: List[str]) -> Dict:
        """
        Check if chunk contains pure/near-pure biblical quotation.
        
        Args:
            chunk_tokens: List of tokens (lemmas for BDC, words for PSC)
        
        Returns:
            Dict with:
                - is_biblical: bool
                - max_overlap: float
                - matching_verse: str (if biblical)
        """
        # remove punctuation, lowercase
        clean_tokens = [t.lower() for t in chunk_tokens if t and t.isalpha()]
        
        if not clean_tokens:
            return {
                "is_biblical": False,
                "max_overlap": 0.0,
                "matching_verse": None
            }
        
        chunk_set = set(clean_tokens)
        
        # Find max overlap across all Vulgate verses
        max_overlap = 0.0
        matching_verse = None
        
        for verse_id, verse_set in self.vulgate_sets.items():
            overlap = self.overlap_coefficient(chunk_set, verse_set)
            
            if overlap > max_overlap:
                max_overlap = overlap
                matching_verse = verse_id
        
        is_biblical = max_overlap >= self.threshold
        
        return {
            "is_biblical": is_biblical,
            "max_overlap": round(max_overlap, 4),
            "matching_verse": matching_verse if is_biblical else None
        }
    
    def filter_corpus(self, chunks: List[Dict], corpus_name: str, use_lemmas: bool = True) -> Dict:
        """
        Filter entire corpus and return filtered results.
        
        Args:
            chunks: List of chunk dictionaries
            corpus_name: Name for logging (BDC/PSC)
            use_lemmas: If True, use 'lemmatized' field; else tokenize 'text'
        
        Returns:
            Dict with filtered_chunks, biblical_chunks, and stats
        """
        logger.info(f"Filtering {corpus_name} ({len(chunks)} chunks)...")
        logger.info(f"Using {'lemmatized' if use_lemmas else 'text'} field")
        logger.info(f"Overlap threshold: {self.threshold}")
        logger.info(f"Method: Overlap Coefficient (Szymkiewicz–Simpson)")
        logger.info(f"Filters chunks where overlap ≥{self.threshold*100:.0f}% with any Bible verse\n")
        
        filtered_chunks = []
        biblical_chunks = []
        
        for chunk in tqdm(chunks, desc=f"Filtering {corpus_name}"):
            # Get tokens based on corpus type
            if use_lemmas:
                tokens = chunk.get("lemmatized", [])
            else:
                # PSC: tokenize text on whitespace
                tokens = chunk.get("text", "").split()
            
            # Compute biblical overlap
            result = self.filter_chunk(tokens)
            
            # Add filtering metadata to chunk
            chunk_with_metadata = chunk.copy()
            chunk_with_metadata.update(result)
            
            # Categorize
            if result["is_biblical"]:
                biblical_chunks.append(chunk_with_metadata)
            else:
                filtered_chunks.append(chunk_with_metadata)
        
        # Compute statistics
        stats = FilterStats(
            total_chunks=len(chunks),
            biblical_chunks=len(biblical_chunks),
            filtered_chunks=len(filtered_chunks)
        )
        
        # Log results
        logger.info(f"\n{corpus_name} FILTERING RESULTS:")
        logger.info(f"  Total chunks: {stats.total_chunks:,}")
        logger.info(f"  Biblical (filtered out): {stats.biblical_chunks:,} ({stats.biblical_percentage:.1f}%)")
        logger.info(f"  Retained: {stats.filtered_chunks:,} ({stats.retention_rate*100:.1f}%)")
        logger.info("")
        
        return {
            "filtered_chunks": filtered_chunks,
            "biblical_chunks": biblical_chunks,
            "stats": {
                "total": stats.total_chunks,
                "biblical": stats.biblical_chunks,
                "retained": stats.filtered_chunks,
                "biblical_percentage": stats.biblical_percentage,
                "retention_rate": stats.retention_rate,
            }
        }


def load_chunks(filepath: Path) -> List[Dict]:
    """Load chunks from JSON file."""
    logger.info(f"Loading {filepath.name}...")
    with open(filepath, 'r', encoding='utf-8') as f:
        data = json.load(f)
    chunks = data.get("chunks", [])
    logger.info(f"✓ Loaded {len(chunks):,} chunks\n")
    return chunks


def save_filtered_data_jsonl(output_dir: Path, filtered_data: Dict, corpus_name: str):
    """
    Save filtered chunks to JSONL
    """
    # Save metadata as JSON
    metadata_path = output_dir / f"{corpus_name.lower()}_filtered_metadata.json"
    metadata = {
        "corpus": corpus_name,
        "filtering": {
            "method": "Overlap Coefficient (Szymkiewicz-Simpson)",
            "threshold": OVERLAP_THRESHOLD,
            "description": f"Filters chunks where overlap ≥{OVERLAP_THRESHOLD*100:.0f}% with any Bible verse",
            "formula": "|chunk ∩ verse| / min(|chunk|, |verse|)",
            "total_chunks": filtered_data["stats"]["total"],
            "biblical_chunks": filtered_data["stats"]["biblical"],
            "retained_chunks": filtered_data["stats"]["retained"],
            "retention_rate": filtered_data["stats"]["retention_rate"],
            "biblical_percentage": filtered_data["stats"]["biblical_percentage"],
        }
    }
    
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2)
    
    logger.info(f"✓ Saved metadata to {metadata_path}")
    
    # Save filtered chunks as JSONL (one chunk per line)
    filtered_path = output_dir / f"{corpus_name.lower()}_filtered.jsonl"
    with open(filtered_path, 'w', encoding='utf-8') as f:
        for chunk in filtered_data["filtered_chunks"]:
            f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
    
    logger.info(f"✓ Saved {len(filtered_data['filtered_chunks']):,} filtered chunks to {filtered_path}")
    
    # Save biblical chunks as JSONL (for reference/analysis)
    biblical_path = output_dir / f"{corpus_name.lower()}_biblical.jsonl"
    with open(biblical_path, 'w', encoding='utf-8') as f:
        for chunk in filtered_data["biblical_chunks"]:
            f.write(json.dumps(chunk, ensure_ascii=False) + '\n')
    
    logger.info(f"✓ Saved {len(filtered_data['biblical_chunks']):,} biblical chunks to {biblical_path}\n")


def main():
    """Main filtering pipeline."""
    
    logger.info("=" * 60)
    logger.info("BIBLICAL CONTENT FILTERING PIPELINE")
    logger.info("Overlap Coefficient (Szymkiewicz-Simpson)")
    logger.info("=" * 60 + "\n")
    
    # Define paths
    vulgate_path = DATA_DIR / "vulgate_chunks.json"
    bdc_path = DATA_DIR / "VD_bdc_chunks.json"
    psc_path = DATA_DIR / "VD_psc_chunks.json"
    
    output_dir = DATA_DIR / "filtered"
    output_dir.mkdir(exist_ok=True)
    
    # Load data
    vulgate_chunks = load_chunks(vulgate_path)
    bdc_chunks = load_chunks(bdc_path)
    psc_chunks = load_chunks(psc_path)
    
    # Initialize filter
    biblical_filter = BiblicalFilter(vulgate_chunks, threshold=OVERLAP_THRESHOLD)
    
    # Filter BDC (use lemmatized field)
    logger.info("-" * 60)
    logger.info("FILTERING BDC (Bullinger Digital Corpus)")
    logger.info("-" * 60 + "\n")
    
    bdc_filtered = biblical_filter.filter_corpus(
        bdc_chunks, 
        corpus_name="BDC",
        use_lemmas=True  # BDC has lemmatized field
    )
    
    save_filtered_data_jsonl(
        output_dir,
        bdc_filtered,
        "BDC"
    )
    
    # Filter PSC (tokenize text on whitespace)
    logger.info("-" * 60)
    logger.info("FILTERING PSC (Patristic Source Corpus)")
    logger.info("-" * 60 + "\n")
    
    psc_filtered = biblical_filter.filter_corpus(
        psc_chunks,
        corpus_name="PSC", 
        use_lemmas=False  # PSC: tokenize text field
    )
    
    save_filtered_data_jsonl(
        output_dir,
        psc_filtered,
        "PSC"
    )


    logger.info(f"\nBDC: {bdc_filtered['stats']['retained']:,} / {bdc_filtered['stats']['total']:,} retained")
    logger.info(f"PSC: {psc_filtered['stats']['retained']:,} / {psc_filtered['stats']['total']:,} retained")


if __name__ == "__main__":
    main()

INFO: ============================================================
INFO: BIBLICAL CONTENT FILTERING PIPELINE
INFO: Overlap Coefficient (Szymkiewicz-Simpson)
INFO: ============================================================

INFO: Loading vulgate_chunks.json...
INFO: ✓ Loaded 35,809 chunks

INFO: Loading VD_bdc_chunks.json...
INFO: ✓ Loaded 19,466 chunks

INFO: Loading VD_psc_chunks.json...
INFO: ✓ Loaded 46,408 chunks

INFO: Pre-computing Vulgate lemma sets from 35809 verses...
Building Vulgate index: 100%|██████████| 35809/35809 [00:00<00:00, 69705.54it/s] 
INFO: ✓ Indexed 35809 Vulgate verses

INFO: ------------------------------------------------------------
INFO: FILTERING BDC (Bullinger Digital Corpus)
INFO: ------------------------------------------------------------

INFO: Filtering BDC (19466 chunks)...
INFO: Using lemmatized field
INFO: Overlap threshold: 0.75
INFO: Method: Overlap Coefficient (Szymkiewicz–Simpson)
INFO: Filters chunks where overlap ≥75% with any Bible verse
